# SCC: OpenMP vs CUDA Benchmark (Real SuiteSparse Graphs)

This notebook downloads **real graphs from the SuiteSparse Matrix Collection**,
the same kind the original OpenMP benchmark used, and runs both OpenMP and CUDA
on them for comparison.

**Requires GPU runtime:** Runtime → Change runtime type → T4 GPU

In [ ]:
# @title 1. Clone repo and check environment
import os, sys, subprocess, time, re, urllib.request, tarfile, glob

REPO_URL = "https://github.com/ShashwaTTrigunayaT/DynamicGraphs_SCC.git"
REPO_DIR = "/content/DynamicGraphs_SCC"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only  # get latest changes
%cd {REPO_DIR}

!echo "=== GPU Check ==="
!nvidia-smi 2>&1 | head -10
!echo "=== CUDA ==="
!nvcc --version 2>&1
!echo "=== CPUs ==="
!nproc

In [ ]:
# @title 2. Fix CUDA arch and build both binaries
# Fix Makefile for Colab T4 (sm_75)
with open("src_CUDA/Makefile", "r") as f:
    content = f.read()
content = content.replace("-arch=sm_70", "-arch=sm_75")
with open("src_CUDA/Makefile", "w") as f:
    f.write(content)
print("Patched Makefile: sm_70 → sm_75")

# Build OpenMP
print("\n=== Building OpenMP binary ===")
%cd /content/DynamicGraphs_SCC/src
!make -s clean 2>&1; make 2>&1 | tail -10
assert os.path.exists("../scc"), "OpenMP build failed!"
print("✓ OpenMP binary built")

# Build CUDA
print("\n=== Building CUDA binary ===")
%cd /content/DynamicGraphs_SCC/src_CUDA
!make -s clean 2>&1; make 2>&1 | tail -10
assert os.path.exists("scc_cuda"), "CUDA build failed!"
print("✓ CUDA binary built")

BINARY_OMP  = "/content/DynamicGraphs_SCC/scc"
BINARY_CUDA = "/content/DynamicGraphs_SCC/src_CUDA/scc_cuda"

In [ ]:
# @title 3. Download real SuiteSparse graphs
# Real directed graphs from SuiteSparse Matrix Collection
# Each: (group, name, description)
GRAPHS = [
    ("HB", "west0067", "67 nodes, 207 edges — chemical engineering"),
    ("HB", "dwt_72",   "72 nodes, 276 edges — structural"),
    ("HB", "can_229",  "229 nodes, 774 edges — structural"),
    ("HB", "ash958",   "958 nodes, 3,068 edges — chemical engineering"),
    ("HB", "mbeacxc",  "496 nodes, 22,710 edges — economics"),
    ("HB", "lshp1009", "1,009 nodes, 3,597 edges — structural"),
    ("HB", "bcsstk01", "48 nodes, 400 edges — stiffness matrix"),
]

datasets_dir = "/content/suite_sparse_graphs"
os.makedirs(datasets_dir, exist_ok=True)

def download_mtx(group, name):
    """Download .mtx file from SuiteSparse and convert to refined_edges.txt."""
    graph_dir = os.path.join(datasets_dir, name)
    os.makedirs(graph_dir, exist_ok=True)
    
    mtx_file = os.path.join(graph_dir, f"{name}.mtx")
    if os.path.exists(mtx_file):
        return mtx_file
    
    url = f"https://sparse.tamu.edu/MM/{group}/{name}.tar.gz"
    tar_path = f"/tmp/{name}.tar.gz"
    
    print(f"  Downloading {group}/{name}...", end=" ", flush=True)
    try:
        urllib.request.urlretrieve(url, tar_path)
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=graph_dir)
        # Find the .mtx file (might be in a subdir)
        mtx_files = glob.glob(os.path.join(graph_dir, "**", "*.mtx"), recursive=True)
        if mtx_files:
            import shutil
            shutil.move(mtx_files[0], mtx_file)
            # Clean up empty subdirectory
            subdir = os.path.dirname(mtx_files[0])
            if subdir and subdir != graph_dir:
                try: shutil.rmtree(subdir)
                except: pass
        print("done")
        return mtx_file
    except Exception as e:
        print(f"FAILED: {e}")
        return None

def mtx_to_edge_list(mtx_path, output_path):
    """Convert .mtx to refined_edges.txt (same logic as dataset_handler.py)."""
    edges = []
    max_v = 0
    header_read = False
    
    with open(mtx_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('%'):
                continue
            parts = line.split()
            if not header_read:
                # First data line: num_rows num_cols num_entries
                header_read = True
                continue
            # Edge line: row col [value]
            u = int(parts[0]) - 1  # 1-indexed → 0-indexed
            v = int(parts[1]) - 1
            if u != v:  # Skip self-loops (same as dataset_handler.py)
                edges.append((u, v))
                max_v = max(max_v, u, v)
    
    with open(output_path, 'w') as f:
        for u, v in edges:
            f.write(f"{u+1} {v+1}\n")  # 1-indexed for OpenMP read_file()
    
    n = max_v + 1  # total nodes (0-indexed max + 1)
    return n, len(edges)

print(f"Downloading {len(GRAPHS)} graphs from SuiteSparse...")
graph_info = []
for group, name, desc in GRAPHS:
    mtx = download_mtx(group, name)
    if mtx:
        out_path = os.path.join(datasets_dir, name, "refined_edges.txt")
        n, m = mtx_to_edge_list(mtx, out_path)
        graph_info.append((name, n, m, desc, out_path))
        print(f"    {name}: {n} nodes, {m} edges")

print(f"\nDownloaded {len(graph_info)} / {len(GRAPHS)} graphs successfully")

In [ ]:
# @title 4. Benchmark runner
def run_and_parse(binary, graph, threads, method, label):
    """Run binary and return (runtime_ms, scc_count)."""
    result = subprocess.run([binary, graph, str(threads), str(method)],
                           capture_output=True, text=True, timeout=300)
    output = result.stdout + result.stderr
    
    runtime = scc = None
    for line in output.split('\n'):
        m = re.search(r'running_time\(ms\)=([0-9.]+)', line)
        if m: runtime = float(m.group(1))
        m = re.search(r'Total # SCCs = (\d+)', line)
        if m: scc = int(m.group(1))
    return runtime, scc, output

all_results = []

for name, n_nodes, m_edges, desc, graph_path in graph_info:
    print(f"\n{'='*70}")
    print(f"Graph: {name}  ({desc})")
    print(f"{'='*70}")
    
    row = {"name": name, "nodes": n_nodes, "edges": m_edges}
    
    # OpenMP Method 2 (full pipeline - reads edge list)
    omp_rt, omp_scc, omp_out = run_and_parse(BINARY_OMP, graph_path, 8, 2, "OMP")
    row["omp_rt"] = omp_rt
    row["omp_scc"] = omp_scc
    for line in omp_out.split('\n'):
        if 'running_time' in line:
            print(f"  [OMP]  {line.strip()}")
        if 'Total # SCCs' in line:
            print(f"  [OMP]  {line.strip()}")
    
    # CUDA Method 0 (Trim1 only)
    cuda0_rt, cuda0_scc, cuda0_out = run_and_parse(BINARY_CUDA, graph_path, 8, 0, "CUDA0")
    row["cuda0_rt"] = cuda0_rt
    row["cuda0_scc"] = cuda0_scc
    for line in cuda0_out.split('\n'):
        if 'running_time' in line:
            print(f"  [CUDA0] {line.strip()}")
        if 'Total # SCCs' in line:
            print(f"  [CUDA0] {line.strip()}")
    
    # CUDA Method 2 (Trim1 + Trim2)
    cuda2_rt, cuda2_scc, cuda2_out = run_and_parse(BINARY_CUDA, graph_path, 8, 2, "CUDA2")
    row["cuda2_rt"] = cuda2_rt
    row["cuda2_scc"] = cuda2_scc
    for line in cuda2_out.split('\n'):
        if 'running_time' in line:
            print(f"  [CUDA2] {line.strip()}")
        if 'Total # SCCs' in line:
            print(f"  [CUDA2] {line.strip()}")
    
    all_results.append(row)

In [ ]:
# @title 5. Comparison Table
print("=" * 120)
h = f"{'Graph':<15s} {'Nodes':<8s} {'Edges':<10s} | "
h += f"{'OMP M2(ms)':<12s} {'SCC':<8s} | "
h += f"{'CUDA M0(ms)':<12s} {'SCC':<8s} | "
h += f"{'CUDA M2(ms)':<12s} {'SCC':<8s} | "
h += f"{'Spd-M0':<8s} {'Spd-M2':<8s} | {'Match'}"
print(h)
print("-" * 120)

for r in all_results:
    name = r['name']
    n = r['nodes']
    m = r['edges']
    omp_rt = r.get('omp_rt', -1)
    omp_scc = r.get('omp_scc', -1)
    c0_rt = r.get('cuda0_rt', -1)
    c0_scc = r.get('cuda0_scc', -1)
    c2_rt = r.get('cuda2_rt', -1)
    c2_scc = r.get('cuda2_scc', -1)
    
    def fmt_ms(v): return f"{v:.2f}" if v > 0 else "FAIL"
    def fmt_scc(v): return str(v) if v >= 0 else "?"
    def fmt_spd(omp, cuda):
        if omp > 0 and cuda > 0: return f"{omp/cuda:.2f}x"
        return "-"
    
    match = ""
    if omp_scc >= 0 and c0_scc >= 0 and c2_scc >= 0:
        # OMP M2 should find all SCCs; CUDA M2 only finds some
        if omp_scc == c2_scc:
            match = "✓"
        elif c0_scc >= c2_scc:
            match = "≈"  # CUDA M2 found fewer (Trim2 did work)
        else:
            match = "△"
    
    s = f"{name:<15s} {n:<8d} {m:<10d} | "
    s += f"{fmt_ms(omp_rt):<12s} {fmt_scc(omp_scc):<8s} | "
    s += f"{fmt_ms(c0_rt):<12s} {fmt_scc(c0_scc):<8s} | "
    s += f"{fmt_ms(c2_rt):<12s} {fmt_scc(c2_scc):<8s} | "
    s += f"{fmt_spd(omp_rt, c0_rt):<8s} {fmt_spd(omp_rt, c2_rt):<8s} | {match}"
    print(s)

print("-" * 120)
print("OMP M2 = full pipeline: Trim1 + Global BFS + Trim2 + WCC + FB-BFS")
print("CUDA M0 = Trim1 only")
print("CUDA M2 = Trim1 + Trim2 (no WCC/FB-BFS yet)")
print("Spd-M0 = speedup OMP vs CUDA M0, Spd-M2 = speedup OMP vs CUDA M2")

In [ ]:
# @title 6. Download as Excel spreadsheet
import pandas as pd
from google.colab import files

rows = []
for r in all_results:
    omp_rt = r.get("omp_rt", -1)
    c0_rt = r.get("cuda0_rt", -1)
    c2_rt = r.get("cuda2_rt", -1)
    omp_scc = r.get("omp_scc", -1)
    c0_scc = r.get("cuda0_scc", -1)
    c2_scc = r.get("cuda2_scc", -1)
    spd_m0 = round(omp_rt / c0_rt, 2) if omp_rt > 0 and c0_rt > 0 else 0
    spd_m2 = round(omp_rt / c2_rt, 2) if omp_rt > 0 and c2_rt > 0 else 0
    match = "✓" if omp_scc == c2_scc else ("≈" if c0_scc >= c2_scc else "△")
    rows.append({
        "Graph": r["name"],
        "Nodes": r["nodes"],
        "Edges": r["edges"],
        "OMP M2 (ms)": round(omp_rt, 2) if omp_rt > 0 else "FAIL",
        "OMP M2 SCCs": omp_scc,
        "CUDA M0 (ms)": round(c0_rt, 2) if c0_rt > 0 else "FAIL",
        "CUDA M0 SCCs": c0_scc,
        "CUDA M2 (ms)": round(c2_rt, 2) if c2_rt > 0 else "FAIL",
        "CUDA M2 SCCs": c2_scc,
        "Speedup M0": f"{spd_m0:.2f}x" if spd_m0 > 0 else "-",
        "Speedup M2": f"{spd_m2:.2f}x" if spd_m2 > 0 else "-",
        "SCC Match": match,
    })

df = pd.DataFrame(rows)

xlsx_path = "/content/scc_benchmark_results.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Comparison", index=False)
    # Auto-adjust column widths
    ws = writer.sheets["Comparison"]
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = max_len + 3

print(f"Excel exported: {xlsx_path}")
print("Downloading...")
files.download(xlsx_path)
print("✓ Done! Open the .xlsx in Excel or Google Sheets.")
